In [ ]:
import numpy as np
import h5py
import matplotlib.pyplot as plt
import scienceplots
import scipy

import tensorstore as ts

plt.style.use(['nature', 'no-latex'])

In [ ]:
import nibabel as nib
import matplotlib.pyplot as plt
import numpy as np
import scienceplots

plt.style.use(['science', 'no-latex'])

In [ ]:
import re, os
np.array([re.search(r'UPENN-GBM-00006_11_(.*)\.nii\.gz', f).group(1)
          for f in os.listdir(path)
          if re.match(r'UPENN-GBM-00006_11_.*\.nii\.gz', f)])

In [ ]:
import os
from matplotlib import font_manager

path = "/Users/s/Downloads/UPENN-GBM-00006_11"
slice_idx = 60
images = []
titles = []

# Collect slice 60 from every .nii.gz file in the folder
for f in np.array(os.listdir(path))[[3, 4, 2]]:
    if f.endswith(".nii.gz"):
        filepath = os.path.join(path, f)
        print(filepath)
        try:
            x = nib.load(filepath)
            data = x.get_fdata()
            if data.shape[2] > slice_idx:
                images.append(data[:, :, slice_idx].T)
                titles.append(re.search(r'UPENN-GBM-00006_11_(.*)\.nii\.gz', f).group(1))
            else:
                print(f"File {f} does not have slice {slice_idx}. Skipping.")
        except Exception as e:
            print(f"Could not load NIfTI file at {filepath}: {e}")

titles

In [ ]:
titles = ["T1", "T1+C", "T2/FLAIR"]

In [ ]:
mask = nib.load("/Users/s/Downloads/UPENN-GBM-00006_11_segm.nii.gz").get_fdata()

In [ ]:
n_cols = 4
n_rows = 2
fig, axes = plt.subplots(2, 4, figsize=(4 * n_cols, 4 * n_rows), dpi=600)

arial_font = font_manager.FontProperties(family='Arial')

# Letters for subplots
letters = ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h']

# Top row: images
for i in range(n_cols):
    ax = axes[0, i]
    ax.imshow(images[i], cmap='gray')
    # Add image type as white text in the lower left corner, using Arial font (BOTTOM LEFT)
    ax.text(
        0.02, 0.015, titles[i], color='white', fontsize=16,
        ha='left', va='bottom', transform=ax.transAxes,
        bbox=dict(facecolor='black', alpha=1, edgecolor='none', pad=2),
        fontproperties=arial_font
    )
    # Add subplot letter in top left
    ax.text(
        0.01, 0.99, letters[i], color='white', fontsize=24, fontweight='bold',
        ha='left', va='top', transform=ax.transAxes,
        bbox=dict(facecolor='black', alpha=0.7, edgecolor='none', pad=1),
        fontproperties=arial_font
    )
    ax.axis('off')

dti_idx = 2
dti_title = "DTI-FA"
dti_img = images[dti_idx]
mask = nib.load("/Users/s/Downloads/UPENN-GBM-00006_11_segm.nii.gz").get_fdata()

# Smooth the mask to represent a heatmap
mask_heatmap = gaussian_filter(mask.astype(float), sigma=2)
mask_heatmap = (mask_heatmap - np.min(mask_heatmap)) / (np.max(mask_heatmap) - np.min(mask_heatmap))

labels = [1, 4, 2]
mask_titles = ["Necrotic core", "Tumour boundary", "Peritumoural edema"]
for i, label in enumerate(labels):
    ax = axes[1, i]
    ax.imshow(dti_img, cmap='gray')
    mask_to_show = mask[..., slice_idx].T == label
    rgba = np.zeros((*mask_to_show.shape, 4), dtype=float)
    rgba[mask_to_show] = [0.5, 0.0, 0.5, 1]  # purple with alpha=0.7
    ax.imshow(rgba)
    # Add mask title as white text in the lower left corner (BOTTOM LEFT)
    ax.text(
        0.02, 0.015, f"{mask_titles[i]}", color='white', fontsize=16,
        ha='left', va='bottom', transform=ax.transAxes,
        bbox=dict(facecolor='black', alpha=1, edgecolor='none', pad=2),
        fontproperties=arial_font
    )
    # Add subplot letter in top left
    ax.text(
        0.01, 0.99, letters[n_cols + i], color='white', fontsize=24, fontweight='bold',
        ha='left', va='top', transform=ax.transAxes,
        bbox=dict(facecolor='black', alpha=0.7, edgecolor='none', pad=1),
        fontproperties=arial_font
    )
    ax.axis('off')

# Show the mask heatmap in the last panel (axes[1, 3])
ax = axes[1, 3]
ax.imshow(dti_img, cmap='gray', vmin=0, vmax=500)
ax.imshow(mask_heatmap[..., slice_idx].T, cmap='hot', alpha=0.4, interpolation='bilinear')

ax.text(
    0.02, 0.015, "Heatmap", color='white', fontsize=16,
    ha='left', va='bottom', transform=ax.transAxes,
    bbox=dict(facecolor='black', alpha=1, edgecolor='none', pad=0),
    fontproperties=arial_font
)
# Add subplot letter in top left
ax.text(
    0.01, 0.99, letters[-1], color='white', fontsize=24, fontweight='bold',
    ha='left', va='top', transform=ax.transAxes,
    bbox=dict(facecolor='black', alpha=0.7, edgecolor='none', pad=1),
    fontproperties=arial_font
)
ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import gaussian_kde
from matplotlib import font_manager
import numpy as np

# Use Arial font for all text
arial_font = font_manager.FontProperties(family='Arial')

# Create a figure with custom gridspec layout, background black
fig = plt.figure(figsize=(16, 8), dpi=600, facecolor='black')

# Create gridspec: 2 rows, 4 columns
gs = gridspec.GridSpec(2, 4, figure=fig)

# Letters for subplots
letters = ['a', 'b', 'c', 'g', 'd', 'e', 'f', 'h']

mri_names = titles[0:3]

# Assuming you have these variables defined:
# images, titles, mask, slice_idx

# Top row: first 3 brain images (a, b, c)
for i in range(3):
    ax = fig.add_subplot(gs[0, i])
    ax.set_facecolor('black')
    if i == 0:
        ax.imshow(images[i], cmap='gray', vmin=0, vmax=1300)
    else:
        ax.imshow(images[i], cmap='gray')

    # Add image type as white text in the lower left corner
    ax.text(
        0.02, 0.015, titles[i], color='white', fontsize=16,
        ha='left', va='bottom', transform=ax.transAxes,
        bbox=dict(facecolor='black', alpha=1, edgecolor='none', pad=2),
        fontproperties=arial_font
    )

    # Add subplot letter in top left
    ax.text(
        0.01, 0.99, letters[i], color='white', fontsize=24, fontweight='bold',
        ha='left', va='top', transform=ax.transAxes,
        bbox=dict(facecolor='black', alpha=0.7, edgecolor='none', pad=1),
        fontproperties=arial_font
    )
    ax.axis('off')

# Top right (panel d): Combined KDE plots
ax_kde = fig.add_subplot(gs[0, 3])
ax_kde.set_facecolor('black')

# Add subplot letter for KDE panel
ax_kde.text(
    0.01, 0.99, letters[3], color='white', fontsize=24, fontweight='bold',
    ha='left', va='top', transform=ax_kde.transAxes,
    bbox=dict(facecolor='black', alpha=1, edgecolor='none', pad=1),
    fontproperties=arial_font
)

# Colors for KDE plots
color_in = 'white'
color_out = 'white'

inside_mask = (mask[..., 60].T > 0)
outside_mask = ~inside_mask

# Calculate x range for KDE plots
x_min, x_max = 50, 750
x_vals = np.linspace(x_min, x_max, 512)

# Create three KDE subplots within the single panel
kde_gs = gridspec.GridSpecFromSubplotSpec(3, 1, gs[0, 3], hspace=0.1)

for i in range(3):
    ax_sub = fig.add_subplot(kde_gs[i])
    ax_sub.set_facecolor('black')

    img = images[i]
    name = titles[i]

    # Get values inside and outside mask
    vals_in = img[inside_mask]
    vals_in = vals_in[vals_in != 0].ravel()
    vals_out = img[outside_mask]
    vals_out = vals_out[vals_out != 0].ravel()

    # Plot KDE for inside mask (solid line)
    if len(vals_in) > 1:
        kde_in = gaussian_kde(vals_in)
        ax_sub.plot(x_vals, kde_in(x_vals), color=color_in, linewidth=3)

    # Plot KDE for outside mask (dashed line)
    if len(vals_out) > 1:
        kde_out = gaussian_kde(vals_out)
        ax_sub.plot(x_vals, kde_out(x_vals), color=color_out,
                    linewidth=3, linestyle='--')


    # Only add x-label to bottom plot
    if i == 2:
        ax_sub.set_xlabel('MRI Intensity', fontsize=15, fontproperties=arial_font, color='white')

    # Configure spines and ticks
    for spine in ax_sub.spines.values():
        spine.set_linewidth(2)
        spine.set_color('white')
    ax_sub.spines['top'].set_visible(False)
    ax_sub.spines['left'].set_visible(False)
    ax_sub.spines['right'].set_visible(False)

    ax_sub.tick_params(axis='both', labelsize=15, colors='white')
    ax_sub.tick_params(axis='x', which='both', top=False, direction='out',
                       width=2, length=6, color='white')
    ax_sub.tick_params(axis='y', which='both', left=False, right=False,
                       direction='out', width=1.2, color='white')

    # Hide y-tick labels
    ax_sub.set_yticklabels([])

    # Hide x-tick labels for top two plots
    if i < 2:
        ax_sub.set_xticklabels([], fontsize=18)

    # Set x-axis limits
    ax_sub.set_xlim(x_min, x_max)

    # Set tick label properties
    for label in ax_sub.get_xticklabels():
        label.set_fontproperties(arial_font)
        label.set_color('white')
        if i == 2:
            label.set_fontsize(14)

    ax_sub.text(-0.1, 0.002, titles[i], color='white', fontsize=16, fontproperties=arial_font)
    if i == 0:
        ax_sub.text(0.01, 0.0056, letters[3], color='white', fontsize=24, fontweight='bold', fontproperties=arial_font)


# Bottom row: mask overlays (e, f, g) and heatmap (h)
dti_idx = 2
dti_img = images[dti_idx]
plot_idx_for_mask = [1, 1, 2]

labels = [1, 4, 2]
mask_titles = ["Necrotic core (T1+C)", "Tumor boundary (T1+C)", "Peritumoral edema (T2/FLAIR)"]

for i, label in enumerate(labels):
    ax = fig.add_subplot(gs[1, i])
    ax.set_facecolor('black')
    ax.imshow(images[plot_idx_for_mask[i]], cmap='gray')

    mask_to_show = mask[..., slice_idx].T == label
    rgba = np.zeros((*mask_to_show.shape, 4), dtype=float)
    rgba[mask_to_show] = [0.5, 0.0, 0.5, 1]  # purple
    ax.imshow(rgba)

    # Add mask title
    ax.text(
        0.02, 0.015, f"{mask_titles[i]}", color='white', fontsize=16,
        ha='left', va='bottom', transform=ax.transAxes,
        bbox=dict(facecolor='black', alpha=1, edgecolor='none', pad=2),
        fontproperties=arial_font
    )

    # Add subplot letter
    ax.text(
        0.01, 0.99, letters[4 + i], color='white', fontsize=24, fontweight='bold',
        ha='left', va='top', transform=ax.transAxes,
        bbox=dict(facecolor='black', alpha=0.7, edgecolor='none', pad=1),
        fontproperties=arial_font
    )
    ax.axis('off')

# Heatmap panel (h)
ax = fig.add_subplot(gs[1, 3])
ax.set_facecolor('black')
ax.imshow(dti_img, cmap='gray', vmin=0, vmax=500)

# Create and show heatmap (assuming you have mask_heatmap)
from scipy.ndimage import gaussian_filter
mask_heatmap = gaussian_filter(mask.astype(float), sigma=2)
mask_heatmap = (mask_heatmap - np.min(mask_heatmap)) / (np.max(mask_heatmap) - np.min(mask_heatmap))
ax.imshow(mask_heatmap[..., slice_idx].T, cmap='hot', alpha=0.4, interpolation='bilinear')

ax.text(
    0.02, 0.015, "Heatmap (T2/FLAIR)", color='white', fontsize=16,
    ha='left', va='bottom', transform=ax.transAxes,
    bbox=dict(facecolor='black', alpha=1, edgecolor='none', pad=0),
    fontproperties=arial_font
)

ax.text(
    0.01, 0.99, letters[7], color='white', fontsize=24, fontweight='bold',
    ha='left', va='top', transform=ax.transAxes,
    bbox=dict(facecolor='black', alpha=0.7, edgecolor='none', pad=1),
    fontproperties=arial_font
)
ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
fig.savefig("/Users/s/Downloads/figure.svg", format="svg", bbox_inches="tight", facecolor=fig.get_facecolor())

In [ ]:
fig.savefig("/Users/s/Downloads/figure.png", format="png", bbox_inches="tight", facecolor=fig.get_facecolor())


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib import font_manager
import numpy as np

# Use Arial font for all text
arial_font = font_manager.FontProperties(family='Arial')

# Create a figure with custom gridspec layout, background black
fig = plt.figure(figsize=(16, 8), dpi=600, facecolor='black')

# Create gridspec: 2 rows, 4 columns
gs = gridspec.GridSpec(2, 4, figure=fig)

# Letters for subplots
letters = ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h']

mri_names = titles[0:3]

# Assuming you have these variables defined:
# images, titles, mask, slice_idx

# Top row: first 3 brain images (a, b, c)
for i in range(3):
    ax = fig.add_subplot(gs[0, i])
    ax.set_facecolor('black')
    if i == 0:
        ax.imshow(images[i], cmap='gray', vmin=0, vmax=1300)
    else:
        ax.imshow(images[i], cmap='gray')

    # Add image type as white text in the lower left corner
    ax.text(
        0.02, 0.015, titles[i], color='white', fontsize=16,
        ha='left', va='bottom', transform=ax.transAxes,
        bbox=dict(facecolor='black', alpha=1, edgecolor='none', pad=2),
        fontproperties=arial_font
    )

    # Add subplot letter in top left
    ax.text(
        0.01, 0.99, letters[i], color='white', fontsize=24, fontweight='bold',
        ha='left', va='top', transform=ax.transAxes,
        bbox=dict(facecolor='black', alpha=0.7, edgecolor='none', pad=1),
        fontproperties=arial_font
    )
    ax.axis('off')

# Top right (panel d): Combined histogram plots
ax_hist = fig.add_subplot(gs[0, 3])
ax_hist.set_facecolor('black')

# Add subplot letter for histogram panel
ax_hist.text(
    0.01, 0.99, letters[3], color='white', fontsize=24, fontweight='bold',
    ha='left', va='top', transform=ax_hist.transAxes,
    bbox=dict(facecolor='black', alpha=1, edgecolor='none', pad=1),
    fontproperties=arial_font
)

# Colors for histogram plots
color_in = 'white'
color_out = 'white'

inside_mask = (mask[..., 60].T > 0)
outside_mask = ~inside_mask

# Calculate x range for histogram plots
x_min, x_max = 50, 750
bins = np.linspace(x_min, x_max, 50)

# Create three histogram subplots within the single panel
hist_gs = gridspec.GridSpecFromSubplotSpec(3, 1, gs[0, 3], hspace=0.1)

for i in range(3):
    ax_sub = fig.add_subplot(hist_gs[i])
    ax_sub.set_facecolor('black')

    img = images[i]
    name = titles[i]

    # Get values inside and outside mask
    vals_in = img[inside_mask]
    vals_in = vals_in[vals_in != 0].ravel()
    vals_out = img[outside_mask]
    vals_out = vals_out[vals_out != 0].ravel()

    # Plot histogram for inside mask (solid fill)
    if len(vals_in) > 1:
        ax_sub.hist(
            vals_in, bins=bins, color=color_in, alpha=0.7, linewidth=0,
            histtype='stepfilled', label='Inside', density=True
        )

    # Plot histogram for outside mask (dashed outline)
    if len(vals_out) > 1:
        ax_sub.hist(
            vals_out, bins=bins, color=color_out, alpha=0.7, linewidth=2,
            histtype='step', linestyle='--', label='Outside', density=True
        )

    # Only add x-label to bottom plot
    if i == 2:
        ax_sub.set_xlabel('MRI Intensity', fontsize=15, fontproperties=arial_font, color='white')

    # Configure spines and ticks
    for spine in ax_sub.spines.values():
        spine.set_linewidth(2)
        spine.set_color('white')
    ax_sub.spines['top'].set_visible(False)
    ax_sub.spines['left'].set_visible(False)
    ax_sub.spines['right'].set_visible(False)

    ax_sub.tick_params(axis='both', labelsize=15, colors='white')
    ax_sub.tick_params(axis='x', which='both', top=False, direction='out',
                       width=2, length=6, color='white')
    ax_sub.tick_params(axis='y', which='both', left=False, right=False,
                       direction='out', width=1.2, color='white')

    # Hide y-tick labels
    ax_sub.set_yticklabels([])

    # Hide x-tick labels for top two plots
    if i < 2:
        ax_sub.set_xticklabels([], fontsize=18)

    # Set x-axis limits
    ax_sub.set_xlim(x_min, x_max)

    # Set tick label properties
    for label in ax_sub.get_xticklabels():
        label.set_fontproperties(arial_font)
        label.set_color('white')
        if i == 2:
            label.set_fontsize(14)

    ax_sub.text(-0.1, 0.002, titles[i], color='white', fontsize=16, fontproperties=arial_font)
    if i == 0:
        ax_sub.text(0.01, 0.004, "d", color='white', fontsize=24, fontweight='bold', fontproperties=arial_font)

# Bottom row: mask overlays (e, f, g) and heatmap (h)
dti_idx = 2
dti_img = images[dti_idx]

labels = [1, 4, 2]
mask_titles = ["Necrotic core (T2/FLAIR)", "Tumor boundary (T2/FLAIR)", "Peritumoral edema (T2/FLAIR)"]

for i, label in enumerate(labels):
    ax = fig.add_subplot(gs[1, i])
    ax.set_facecolor('black')
    ax.imshow(dti_img, cmap='gray')

    mask_to_show = mask[..., slice_idx].T == label
    rgba = np.zeros((*mask_to_show.shape, 4), dtype=float)
    rgba[mask_to_show] = [0.5, 0.0, 0.5, 1]  # purple
    ax.imshow(rgba)

    # Add mask title
    ax.text(
        0.02, 0.015, f"{mask_titles[i]}", color='white', fontsize=16,
        ha='left', va='bottom', transform=ax.transAxes,
        bbox=dict(facecolor='black', alpha=1, edgecolor='none', pad=2),
        fontproperties=arial_font
    )

    # Add subplot letter
    ax.text(
        0.01, 0.99, letters[4 + i], color='white', fontsize=24, fontweight='bold',
        ha='left', va='top', transform=ax.transAxes,
        bbox=dict(facecolor='black', alpha=0.7, edgecolor='none', pad=1),
        fontproperties=arial_font
    )
    ax.axis('off')

# Heatmap panel (h)
ax = fig.add_subplot(gs[1, 3])
ax.set_facecolor('black')
ax.imshow(dti_img, cmap='gray', vmin=0, vmax=500)

# Create and show heatmap (assuming you have mask_heatmap)
from scipy.ndimage import gaussian_filter
mask_heatmap = gaussian_filter(mask.astype(float), sigma=2)
mask_heatmap = (mask_heatmap - np.min(mask_heatmap)) / (np.max(mask_heatmap) - np.min(mask_heatmap))
ax.imshow(mask_heatmap[..., slice_idx].T, cmap='hot', alpha=0.4, interpolation='bilinear')

ax.text(
    0.02, 0.015, "Heatmap", color='white', fontsize=16,
    ha='left', va='bottom', transform=ax.transAxes,
    bbox=dict(facecolor='black', alpha=1, edgecolor='none', pad=0),
    fontproperties=arial_font
)

ax.text(
    0.01, 0.99, letters[7], color='white', fontsize=24, fontweight='bold',
    ha='left', va='top', transform=ax.transAxes,
    bbox=dict(facecolor='black', alpha=0.7, edgecolor='none', pad=1),
    fontproperties=arial_font
)
ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
plt.imshow(images[1], cmap='gray')

In [ ]:
inside_mask = (mask[..., 60].T == 1)
inside_mask_boundary = (mask[..., 60].T == 4)
mask_edema = (mask[..., 60].T == 2)
outside_mask = (mask[..., 60].T == 0)

In [ ]:
vals_in = images[1][inside_mask]
vals_in = vals_in[vals_in != 0].ravel()
vals_out = images[1][outside_mask]
vals_out = vals_out[vals_out != 0].ravel()
vals_in_boundary = images[1][inside_mask_boundary]
vals_in_boundary = vals_in_boundary[vals_in_boundary != 0].ravel()
vals_edema = images[1][mask_edema]
vals_edema = vals_edema[vals_edema != 0].ravel()

plt.figure(figsize=(10, 10))
plt.hist(vals_in, bins=50, color='blue', alpha=0.7, label='Inside', density=False)
plt.hist(vals_out, bins=50, color='black', alpha=0.7, label='Outside', density=False)
plt.hist(vals_in_boundary, bins=50, color='red', alpha=0.7, label='Boundary', density=False)
plt.hist(vals_edema, bins=50, color='green', alpha=0.7, label='Edema', density=False);

In [ ]:
plt.figure(figsize=(10, 10), dpi=600)
plt.imshow(images[1], cmap='gray')
plt.imshow(((mask[..., 60].T == 4) | (mask[..., 60].T == 2)), alpha=0.1, cmap='hot')